In [13]:
import os
import json
import pandas as pd
import xml.etree.ElementTree as ET
from typing import List, Dict, Optional, Any, TypedDict
from pathlib import Path
import re

# --- Pydantic Imports ---
from pydantic import BaseModel, Field, ConfigDict
from operator import itemgetter

# --- LangGraph Imports ---
from langgraph.graph import StateGraph, END
# from langgraph.graph.message import add_messages


class DatasetMetadata(BaseModel):
    name: str
    path: str
    profile_summary: Optional[Dict[str, Any]] = None
    schema: Optional[List[str]] = None


class IntegrationState(BaseModel):
    """
    Central state shared across all agents.
    Tracks datasets, schema matches, issues, and final fused output.
    """
    model_config = ConfigDict(arbitrary_types_allowed=True)

    datasets: List[DatasetMetadata] = Field(default_factory=list)
    # ... (Other fields are omitted for brevity but assumed present)
    fused_data: Optional[pd.DataFrame] = None

# --- LangGraph State Definition ---
class AgentState(TypedDict):
    """The state of the graph, passed between nodes."""
    input_message: str
    schema_code: str
    resolution_code: str
    validator_output: str
    max_retries: int
    current_retry: int
    validation_status: str # CODE_OK or CODE_FIXED or FAILED

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_fields.py:198: UserWarning: Field name "schema" in "DatasetMetadata" shadows an attribute in parent "BaseModel"
  warnings.warn(


In [15]:
# --- External Imports ---
# from PyDI.profiling import DataProfiler
# from PyDI.io import load_csv, load_xml, load_parquet
from dotenv import load_dotenv

# --- LangChain Imports ---
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
# from langchain_community import hub
from langchain.agents import AgentExecutor, create_tool_calling_agent


load_dotenv()

# --- UTILITY FUNCTION ---
def extract_python_code(response_text: str) -> Optional[str]:
    """Extracts the Python code block from the LLM response."""
    # Use re.DOTALL to match across newlines
    match = re.search(r"```python\s*(.*?)\s*```", response_text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

# --- AGENT TOOLS (Identical to previous versions) ---

@tool
def load_and_profile_dataset(file_path: str) -> str:
    """
    Loads dataset (CSV, Parquet, XML) and profiles it.
    The output is a JSON string summarizing the dataset's profile and a 3-row sample.
    """
    file_path = Path(file_path)
    ext = file_path.suffix.lower()
    df = None
    try:
        if ext == ".csv":
            df = pd.read_csv(file_path)
        elif ext == ".parquet":
            df = pd.read_parquet(file_path)
        elif ext == ".xml":
            tree = ET.parse(file_path)
            root = tree.getroot()
            data = []
            for child in root:
                record = {}
                for elem in child:
                    record[elem.tag] = elem.text
                data.append(record)
            df = pd.DataFrame(data)
        else:
            raise ValueError(f"Unsupported format: {ext}. Supported: CSV, Parquet, XML.")

        if df is None or df.empty:
             return f"Error: Dataset at {file_path} loaded as empty or failed to load."


        file_size_mb = os.path.getsize(str(file_path)) / (1024 * 1024)
        null_ratio = df.isna().sum().sum() / (df.shape[0] * df.shape[1])
        return json.dumps({
            "dataset_name": file_path.stem,
            "file_path": str(file_path),
            "file_type": file_path.suffix.lower().replace(".", ""),
            "num_rows": df.shape[0],
            "num_columns": df.shape[1],
            "columns": list(df.columns),
            "file_size_MB": round(file_size_mb, 2),
            "null_ratio": round(null_ratio, 4),
        }, indent=2)

    except Exception as e:
        return f"Error loading or profiling dataset at {file_path}: {e}"

@tool
def get_pydi_doc(module_name: str) -> str:
    """
    Fetch documentation for a PyDI module or class.
    Example usage: get_pydi_doc('PyDI.schemamatching.SchemaMatcher')
    """
    try:
        components = module_name.split(".")
        module = __import__(components[0])
        for comp in components[1:]:
            module = getattr(module, comp)
        return module.__doc__ or f"No docstring available for {module_name}."
    except Exception as e:
        return f"Error retrieving docs for {module_name}: {e}"

In [16]:
# --- AGENT DEFINITIONS (Identical to previous versions) ---

# Base LLM Definition
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2
)


integration_tools = [load_and_profile_dataset, get_pydi_doc]
# agent_prompt_base = hub.pull("hwchase17/structured-chat-agent")

agent_prompt_base = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI assistant with access to tools."),
        MessagesPlaceholder(variable_name="chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

# 1. SCHEMA INTEGRATION AGENT (Code Generator)
integration_agent_runnable = create_tool_calling_agent(llm, integration_tools, agent_prompt_base)
integration_agent_executor = AgentExecutor(
    agent=integration_agent_runnable,
    tools=integration_tools,
    verbose=False, # Set to False for cleaner LangGraph logs
    handle_parsing_errors=True
)

# 2. CODE VALIDATOR AGENT (Self-Healing Debugger - Reusable)
validator_prompt_template = """
You are an expert Python Code Validator and Debugger.

Your task is to review and fix the PyDI Python script below.

The code generated is:
--- CODE START ---
{generated_code}
--- CODE END ---

**Self-Healing/Fixing Instructions:**
1.  **Critique:** Review the code for all common errors (syntax, logic, missing imports, PyDI method signatures) that would prevent it from running.
2.  **Fix:** If an error is found, rewrite the entire, corrected, fully working Python script.
3.  **Validation:** If the code appears perfect and ready to run, respond **ONLY** with the phrase "CODE_OK".

Respond ONLY with the complete, corrected Python code inside triple backticks (```python ... ```) or the exact phrase "CODE_OK". Do not include any other text or reasoning.
"""
validator_prompt = ChatPromptTemplate.from_template(validator_prompt_template)
validator_chain = validator_prompt | llm | StrOutputParser()

# 3. IDENTITY RESOLUTION AGENT (Blocking & Matching Code Generator)
resolution_prompt_template = """
You are an expert PyDI Entity Resolution Agent specializing in blocking and matching.

A previous agent successfully generated and validated the **Schema Matching** code.
Your goal is to complete the pipeline by generating the code for **Identity Resolution**.
**Only use PyDI library**
**Do not use any other libraries (e.g., pandas) for loading or manipulating data.**

For example:
* For loading the datasets, use PyDI.io.load_csv or PyDI.io.load_parquet.
* For comparisons, use PyDI.entitymatching.StringComparator or PyDI.entitymatching.NumericComparator.
* For blocking and matching, use PyDI.entitymatching.StandardBlocker or PyDI.entitymatching.RuleBasedMatcher.


**Context (Validated Schema Matching Code):**
--- PREVIOUS CODE START ---
{validated_schema_match_code}
--- PREVIOUS CODE END ---

**Your Task:**
1.  **Assume** the code in the 'PREVIOUS CODE' block runs successfully and variables like `df_a`, `df_b`, and the schema mapping result are available.
2.  **Use your PyDI knowledge and the 'get_pydi_doc' tool** to determine the optimal blocking and matching methods (e.g., `StandardBlocker`, `SortedNeighbourhoodBlocker`, `TokenBlocker`) based on the assumed schema.
3.  **Generate a complete, fully executable PyDI Python script** that performs the *entire* workflow:
    * Imports (from PyDI.io, PyDI.profiling, PyDI.entitymatching, etc.)
    * Dataset Loading (using the provided paths)
    * **Schema Matching (Integrate the validated code)**
    * **Blocking (Blocking setup and execution)**
    * **Matching/Pairing (Matcher setup and execution)**

Respond ONLY with the complete, end-to-end Python script inside triple backticks.
"""
resolution_agent_runnable = create_tool_calling_agent(llm, integration_tools, agent_prompt_base.partial(
    system=resolution_prompt_template
))
resolution_agent_executor = AgentExecutor(
    agent=resolution_agent_runnable,
    tools=integration_tools,
    verbose=False, # Set to False for cleaner LangGraph logs
    handle_parsing_errors=True
)

In [17]:
# Define the task prompt for the Integration Agent
integration_task_template = """
You are an expert Autonomous Schema Integration Agent specializing in data
manipulation and integration using the PyDI library. **Only use PyDI library**
**Do not use any other libraries (e.g., pandas) for loading or manipulating data.**

For example:
* For loading the datasets, use PyDI.io.load_csv or PyDI.io.load_parquet.
* For schema matching, use PyDI.schemamatching.BaseSchemaMatcher.

**Your Task:**

**1. Primary Goal:**
Your overarching task is to produce a single, minimal, and fully functional
**PyDI Python script** that performs schema matching between the provided datasets.

**2. Execution Steps (Follow this order):**
1.  **Analyze Inputs:** Use `load_and_profile_dataset` on **Dataset A**, and **Dataset B**
    to understand their structure, columns, and data types.
2.  **Research:** Use `get_pydi_doc` to look up necessary PyDI modules, particularly
    for schema matching (e.g., PyDI.schemamatching or PyDI.entitymatching).
3.  **Code Generation:** Based on the dataset profiles and PyDI documentation,
    write the complete Python code for schema matching.

**3. Datasets to Use:**
* Datasets: {datasets}

**4. Final Output Format:**
* Your final, complete response MUST contain ONLY the Python code
    required to load the datasets, define the schema matching process using PyDI,
    and execute the matching.
* The code must be enclosed **ONLY within a single set of triple backticks** (```python ... ```).
* Do NOT include any explanatory text, commentary, or conversational dialogue outside of the code block.

Begin by calling your tool to profile the first dataset mentioned.
"""

integration_agent_runnable = create_tool_calling_agent(llm, integration_tools, agent_prompt_base.partial(
    system=integration_task_template
))
integration_agent_executor = AgentExecutor(
    agent=integration_agent_runnable,
    tools=integration_tools,
    verbose=False, # Set to False for cleaner LangGraph logs
    handle_parsing_errors=True
)

In [18]:
# --- LANGGRAPH NODE FUNCTIONS ---

def generate_schema_code(state: AgentState) -> dict:
    """Node 1: Generates initial Schema Matching code."""
    print(f"\n[NODE] Running Schema Generator (Attempt {state.get('current_retry', 1)})")

    integration_task_description = integration_task_template.format(
        datasets=state['input_message']
    )

    response = integration_agent_executor.invoke({"input": integration_task_description})
    initial_code = extract_python_code(response.get("output", ""))

    return {"schema_code": initial_code or "Failed to generate code."}


def validate_schema_code(state: AgentState) -> dict:
    """Node 2: Validates the generated Schema Matching code."""
    print("\n[NODE] Running Validator on Schema Code...")
    code_to_validate = state["schema_code"]

    if not code_to_validate or "Failed to generate code." in code_to_validate:
        return {"validation_status": "FAILED", "validator_output": "No code generated."}

    validation_input = {"generated_code": code_to_validate}
    validation_result = validator_chain.invoke(validation_input)

    if validation_result.strip() == "CODE_OK":
        print("[VALIDATOR] Schema Code Confirmed: CODE_OK")
        return {"validation_status": "CODE_OK", "validator_output": "CODE_OK", "current_retry": 0}

    fixed_code = extract_python_code(validation_result)

    if fixed_code:
        print("[VALIDATOR] Schema Code Fixed: Retrying generation step with fixed code as context.")
        return {
            "validation_status": "CODE_FIXED",
            "schema_code": fixed_code,
            "current_retry": state.get('current_retry', 1) + 1
        }

    print("[VALIDATOR] Schema Code Validation Failed: No clean fix provided.")
    return {"validation_status": "FAILED"}


def generate_resolution_code(state: AgentState) -> dict:
    """Node 3: Generates Identity Resolution code using validated Schema code."""
    print("\n[NODE] Running Resolution Generator.")

    resolution_prompt_context = resolution_prompt_template.format(
        validated_schema_match_code=state["schema_code"]
    )

    response = resolution_agent_executor.invoke({"input": resolution_prompt_context})
    resolution_code = extract_python_code(response.get("output", ""))

    return {"resolution_code": resolution_code or "Failed to generate resolution code."}


def validate_resolution_code(state: AgentState) -> dict:
    """Node 4: Validates the generated Identity Resolution code."""
    print("\n[NODE] Running Validator on Resolution Code...")
    code_to_validate = state["resolution_code"]

    if not code_to_validate or "Failed to generate resolution code." in code_to_validate:
        return {"validation_status": "FAILED", "validator_output": "No resolution code generated."}

    validation_input = {"generated_code": code_to_validate}
    validation_result = validator_chain.invoke(validation_input)

    if validation_result.strip() == "CODE_OK":
        print("[VALIDATOR] Resolution Code Confirmed: CODE_OK")
        return {"validation_status": "CODE_OK", "validator_output": "CODE_OK"}

    fixed_code = extract_python_code(validation_result)

    # We use a special key here to track retries for the resolution phase separately
    resolution_retries = state.get('resolution_retries', 0) + 1

    if fixed_code and resolution_retries <= state['max_retries']:
        print(f"[VALIDATOR] Resolution Code Fixed (Attempt {resolution_retries}/{state['max_retries']}): Retrying resolution generation.")
        return {
            "validation_status": "CODE_FIXED",
            "resolution_code": fixed_code,
            "resolution_retries": resolution_retries
        }

    print("[VALIDATOR] Resolution Code Validation Failed: Max retries reached or no clean fix.")
    return {"validation_status": "FAILED"}

In [20]:
# --- LANGGRAPH CONDITIONAL EDGES ---

def decide_schema_fix(state: AgentState) -> str:
    """Decide whether to fix/retry Schema generation or move to Resolution."""
    status = state["validation_status"]
    current_retry = state.get('current_retry', 1)

    if status == "CODE_OK":
        print("[DECISION] Schema Code Validated. Moving to Resolution Generator.")
        return "generate_resolution_code"

    if status == "CODE_FIXED" and current_retry < state["max_retries"]:
        print(f"[DECISION] Schema Code Fixed. Retrying Schema Generator (Attempt {current_retry}).")
        return "generate_schema_code"

    print("[DECISION] Schema Code Failed (Max retries/Failure). Ending pipeline.")
    return END


def decide_resolution_fix(state: AgentState) -> str:
    """Decide whether to fix/retry Resolution generation or finish."""
    status = state["validation_status"]
    resolution_retries = state.get('resolution_retries', 1)

    if status == "CODE_OK":
        print("[DECISION] Resolution Code Validated. Ending pipeline.")
        return END

    if status == "CODE_FIXED" and resolution_retries < state["max_retries"]:
        print(f"[DECISION] Resolution Code Fixed. Retrying Resolution Generator (Attempt {resolution_retries}).")
        return "generate_resolution_code"

    print("[DECISION] Resolution Code Failed (Max retries/Failure). Ending pipeline.")
    return END

# --- ASSEMBLE THE GRAPH ---

graph = StateGraph(AgentState)

# Add Nodes
graph.add_node("generate_schema_code", generate_schema_code)
graph.add_node("validate_schema_code", validate_schema_code)
graph.add_node("generate_resolution_code", generate_resolution_code)
graph.add_node("validate_resolution_code", validate_resolution_code)

# Build the Schema Flow (with self-healing loop)
graph.set_entry_point("generate_schema_code")
graph.add_edge("generate_schema_code", "validate_schema_code")
graph.add_conditional_edges("validate_schema_code", decide_schema_fix)

# Build the Resolution Flow (with self-healing loop)
graph.add_edge("generate_resolution_code", "validate_resolution_code")
graph.add_conditional_edges("validate_resolution_code", decide_resolution_fix)

# Compile the Graph
app = graph.compile()


# --- EXECUTION ---

TASK_INPUT = {
    # "dataset_a_path": "datasets/amazon.patquet",
    "dataset_a_path": "datasets/goodreads.parquet",
    "dataset_b_path": "datasets/metabooks.parquet"
}

initial_state: AgentState = {
    "input_message": f"Datasets: {TASK_INPUT['dataset_a_path']} and {TASK_INPUT['dataset_b_path']}",
    "schema_code": "",
    "resolution_code": "",
    "validator_output": "",
    "max_retries": 3, # Maximum number of times to try fixing code
    "current_retry": 1,
    "validation_status": "",
    "resolution_retries": 1,
}

print("--- STARTING LANGGRAPH EXECUTION ---")
print("Target: Schema Matching and Identity Resolution PyDI Pipeline.")

# Run the graph
final_state = app.invoke(initial_state)

print(f"Final Schema Code Status: {final_state['validation_status']}")

# Check if the pipeline completed successfully
if final_state["validation_status"] == "CODE_OK" and final_state["resolution_code"]:
    print("\nFINAL VALIDATED PYDI PIPELINE SCRIPT:")
    print("--"*50)
    print(final_state['resolution_code'])
    print("--"*50)
else:
    print("\n[FAILURE] Pipeline failed to produce validated Identity Resolution code after max retries.")
    print("Last generated Schema Code:")
    print(final_state['schema_code'])
    print("\nLast generated Resolution Code (May be incomplete/invalid):")
    print(final_state['resolution_code'])

--- STARTING LANGGRAPH EXECUTION ---
Target: Schema Matching and Identity Resolution PyDI Pipeline.

[NODE] Running Schema Generator (Attempt 1)

[NODE] Running Validator on Schema Code...
[VALIDATOR] Schema Code Fixed: Retrying generation step with fixed code as context.
[DECISION] Schema Code Fixed. Retrying Schema Generator (Attempt 2).

[NODE] Running Schema Generator (Attempt 2)

[NODE] Running Validator on Schema Code...
[VALIDATOR] Schema Code Fixed: Retrying generation step with fixed code as context.
[DECISION] Schema Code Failed (Max retries/Failure). Ending pipeline.
Final Schema Code Status: CODE_FIXED

[FAILURE] Pipeline failed to produce validated Identity Resolution code after max retries.
Last generated Schema Code:
import PyDI
from PyDI.io import load_parquet
from PyDI.schemamatching import BaseSchemaMatcher

# Load datasets
try:
    dataset_a = load_parquet('/content/datasets/goodreads.parquet')
    dataset_b = load_parquet('/content/datasets/metabooks.parquet')
excep